In [ ]:
import os
import numpy as np
import pandas as pd

HUBS = [
    ("HUB01", "Mumbai", "West", 1, 1500), ("HUB02", "Delhi", "North", 1, 1600),
    ("HUB03", "Bengaluru", "South", 1, 1450), ("HUB04", "Hyderabad", "South", 1, 1350),
    ("HUB05", "Chennai", "South", 1, 1300), ("HUB06", "Pune", "West", 1, 1250),
    ("HUB07", "Kolkata", "East", 1, 1200), ("HUB08", "Ahmedabad", "West", 1, 1150),
    ("HUB09", "Jaipur", "North", 2, 950), ("HUB10", "Lucknow", "North", 2, 900),
    ("HUB11", "Chandigarh", "North", 2, 850), ("HUB12", "Indore", "West", 2, 820),
    ("HUB13", "Nagpur", "West", 2, 780), ("HUB14", "Coimbatore", "South", 2, 740),
    ("HUB15", "Kochi", "South", 2, 700), ("HUB16", "Surat", "West", 2, 650),
    ("HUB17", "Patna", "East", 3, 520), ("HUB18", "Ranchi", "East", 3, 480),
    ("HUB19", "Guwahati", "East", 3, 450), ("HUB20", "Raipur", "East", 3, 420),
    ("HUB21", "Bhubaneswar", "East", 3, 390), ("HUB22", "Dehradun", "North", 3, 350),
    ("HUB23", "Siliguri", "East", 3, 310), ("HUB24", "Agartala", "East", 3, 270),
]
DIST_MU = [180, 320, 480]      # mean lane distance (km) for hub tier 1 / 2 / 3
DIST_SD = [50, 90, 150]
STOPS_MU = [55, 70, 85]        # mean stops on the rider route for hub tier 1 / 2 / 3
HUB_EFFECT = [0, 2, 4]         # extra hours for hub tier 1 / 2 / 3
# How a hub really operates (1 = metro-like, 2 = regional, 3 = stretched). Four hubs operate unlike their tier label.
OPERATING_TIER = {"HUB07": 2, "HUB13": 3, "HUB15": 3, "HUB22": 2}
BASE_HOURS = {"Express": 10, "Standard": 28, "Economy": 58}


def simulate(n_per_hub, seed):
    """Clean shipments, n_per_hub rows for each of the 24 hubs, in hub order."""
    np.random.seed(seed)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    n = n_per_hub * len(hubs)
    op_tier = [OPERATING_TIER.get(h, t) for h, t in zip(hubs["hub_id"], hubs["hub_tier"])]
    tier = np.repeat(op_tier, n_per_hub) - 1
    hub_id = np.repeat(hubs["hub_id"].values, n_per_hub)
    distance = np.clip(np.random.normal(np.array(DIST_MU)[tier], np.array(DIST_SD)[tier]), 15, 1500).round(1)
    stops = np.clip(np.random.normal(np.array(STOPS_MU)[tier], 10), 20, 140).round().astype(int)
    weight = np.clip(np.random.lognormal(0.6, 0.7, n), 0.1, 40).round(2)
    value = np.random.lognormal(7.0, 0.8, n).round().astype(int)
    service = np.random.choice(["Express", "Standard", "Economy"], n, p=[0.2, 0.55, 0.25])
    pay = np.random.choice(["Prepaid", "COD"], n, p=[0.6, 0.4])
    rain = (np.random.rand(n) < 0.18).astype(int)
    day = np.random.randint(0, 90, n)
    hour = np.random.randint(6, 22, n)
    minute = np.random.randint(0, 60, n)
    ts = pd.Timestamp("2026-07-01") + pd.to_timedelta(day, unit="D") + pd.to_timedelta(hour * 60 + minute, unit="m")
    sale = ((ts >= pd.Timestamp("2026-09-15")) & (ts < pd.Timestamp("2026-09-29"))).astype(int)
    weekend = (ts.dayofweek >= 5).astype(int)
    hours = (np.array([BASE_HOURS[s] for s in service]) + 0.03 * distance + 0.10 * stops
             + np.array(HUB_EFFECT)[tier] + 0.12 * weight + 4 * rain + 7 * sale
             + 3 * (pay == "COD") + 2 * weekend + 3 * (hour >= 18)
             + 12 * sale * (service == "Express") + 8 * (distance > 400) * rain
             + np.random.normal(0, 4, n)).round(2)
    return pd.DataFrame({
        "hub_id": hub_id, "service_type": service, "payment_mode": pay,
        "parcel_weight_kg": weight, "distance_km": distance, "declared_value": value,
        "n_stops": stops, "rain_flag": rain, "sale_event_flag": sale, "dispatch_time": ts,
        "label_print_batch": np.random.randint(1000, 9999, n),
        "scanner_build_no": np.random.randint(1, 40, n),
        "tape_roll_no": np.random.randint(1, 500, n),
        "sticker_serial_mod": np.random.uniform(0, 1, n).round(3),
        "shelf_slot_no": np.random.randint(1, 300, n),
        "delivery_hours": hours,
    })


def make_messy(df, seed, n_missing, n_case, n_sentinel, n_dupes, id_start):
    """Turn a clean table into a raw export: text columns, gaps, duplicates, a sentinel."""
    np.random.seed(seed)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    df.insert(0, "shipment_id", ["SHP" + str(id_start + i) for i in range(n)])
    df["dispatch_time"] = df["dispatch_time"].dt.strftime("%Y-%m-%d %H:%M")
    df["declared_value"] = df["declared_value"].map(lambda v: "\u20b9" + format(v, ","))
    for col, k in [("distance_km", n_missing[0]), ("parcel_weight_kg", n_missing[1]),
                   ("declared_value", n_missing[2]), ("payment_mode", n_missing[3])]:
        df.loc[np.random.choice(n, k, replace=False), col] = np.nan
    pick = np.random.choice(n, n_case, replace=False)
    shapes = np.random.randint(0, 3, n_case)
    for i, s in zip(pick, shapes):
        v = df.at[i, "service_type"]
        df.at[i, "service_type"] = [v.upper(), v.lower(), "  " + v + " "][s]
    if n_sentinel:
        df.loc[np.random.choice(n, n_sentinel, replace=False), "parcel_weight_kg"] = 999.0
    if n_dupes:
        df = pd.concat([df, df.iloc[np.random.choice(n, n_dupes, replace=False)]], ignore_index=True)
    return df


if __name__ == "__main__":
    os.makedirs("data", exist_ok=True)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    hubs.to_csv("data/hubs.csv", index=False)
    raw = make_messy(simulate(50, 2604), seed=7, n_missing=(36, 24, 12, 18),
                     n_case=120, n_sentinel=5, n_dupes=24, id_start=10001)
    raw.to_csv("data/raw_shipments.csv", index=False)
    new = make_messy(simulate(2, 99), seed=8, n_missing=(3, 2, 1, 2),
                     n_case=6, n_sentinel=0, n_dupes=0, id_start=90001)
    new = new.drop(columns=["delivery_hours"])
    new.to_csv("data/new_shipments.csv", index=False)
    print(len(hubs), len(raw), len(new))

24 1224 48


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
hubs_df = pd.read_csv("/content/data/hubs.csv")
hubs_df.head()

,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300


In [ ]:
hubs_df.shape

(24, 5)

In [ ]:
raw_ship_df = pd.read_csv("/content/data/raw_shipments.csv")
raw_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [ ]:
new_ship_df = pd.read_csv("/content/data/new_shipments.csv")
new_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no
0,SHP90001,HUB22,Standard,Prepaid,1.52,478.7,₹483,78,0,0,2026-08-03 17:26,9034,5,311,0.710,273
1,SHP90002,HUB24,Standard,COD,0.76,405.5,"₹1,137",94,1,0,2026-08-21 12:52,7336,8,163,0.784,284
2,SHP90003,HUB18,Standard,Prepaid,2.00,552.4,₹195,99,0,0,2026-08-07 16:29,9987,3,291,0.927,140
3,SHP90004,HUB12,Economy,Prepaid,1.66,243.0,"₹1,018",65,0,0,2026-08-28 14:41,4749,14,257,0.405,21
4,SHP90005,HUB22,Express,COD,0.72,219.1,NaN,72,0,0,2026-08-25 18:11,1364,11,442,0.832,164


In [ ]:
PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
 "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value",
"n_stops",
 "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
 "hub_tier", "hub_daily_capacity",
 "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod",
"shelf_slot_no",
]
NOISE_COLUMNS = ["label_print_batch", "scanner_build_no", "tape_roll_no",
"sticker_serial_mod", "shelf_slot_no"]

In [ ]:
PROMISED_HOURS

{'Express': 36, 'Standard': 60, 'Economy': 96}

In [ ]:
FEATURES

['promised_hours',
 'distance_km',
 'parcel_weight_kg',
 'log_declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'is_cod',
 'is_weekend',
 'dispatch_hour',
 'hub_tier',
 'hub_daily_capacity',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no']

In [ ]:
len(FEATURES)

17

In [ ]:
raw_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [ ]:
raw_ship_df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
1219,True
1220,True
1221,True
1222,True


In [ ]:
raw_ship_df.duplicated().sum()

np.int64(24)

In [ ]:
raw_ship_df.duplicated().sum()*100/len(raw_ship_df)

np.float64(1.9607843137254901)

In [ ]:
len(raw_ship_df)

1224

In [ ]:
24*100/1224

1.9607843137254901

In [ ]:
raw_ship_df.duplicated().mean()*100

np.float64(1.9607843137254901)

In [ ]:
raw_ship_df.sample(5, random_state = 42)

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
661,SHP10662,HUB16,Standard,Prepaid,7.81,316.6,"₹1,536",80,0,1,2026-09-25 09:48,2021,12,416,0.006,173,55.16
220,SHP10221,HUB08,Economy,COD,1.94,134.7,"₹1,246",66,1,0,2026-08-19 19:40,9694,15,451,0.627,21,76.38
155,SHP10156,HUB18,standard,Prepaid,2.66,541.0,"₹1,108",80,0,0,2026-07-07 15:02,2947,22,253,0.239,124,54.75
677,SHP10678,HUB03,Standard,Prepaid,0.71,255.5,₹560,58,0,0,2026-08-14 19:59,7558,33,227,0.884,203,44.17
911,SHP10912,HUB04,Standard,COD,3.75,167.5,₹618,66,0,0,2026-08-14 12:38,3127,10,10,0.626,287,44.30


In [ ]:
cols = raw_ship_df.columns.tolist()
cols

['shipment_id',
 'hub_id',
 'service_type',
 'payment_mode',
 'parcel_weight_kg',
 'distance_km',
 'declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'dispatch_time',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no',
 'delivery_hours']

In [ ]:
cols = list(raw_ship_df.columns)
cols

['shipment_id',
 'hub_id',
 'service_type',
 'payment_mode',
 'parcel_weight_kg',
 'distance_km',
 'declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'dispatch_time',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no',
 'delivery_hours']

In [ ]:
raw_ship_df.dtypes

,0
shipment_id,object
hub_id,object
service_type,object
payment_mode,object
parcel_weight_kg,float64
distance_km,float64
declared_value,object
n_stops,int64
rain_flag,int64
sale_event_flag,int64


In [ ]:
raw_ship_df.isna().sum()

,0
shipment_id,0
hub_id,0
service_type,0
payment_mode,18
parcel_weight_kg,25
distance_km,36
declared_value,12
n_stops,0
rain_flag,0
sale_event_flag,0


In [ ]:
raw_ship_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1224 entries, 0 to 1223
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   shipment_id         1224 non-null   object 
 1   hub_id              1224 non-null   object 
 2   service_type        1224 non-null   object 
 3   payment_mode        1206 non-null   object 
 4   parcel_weight_kg    1199 non-null   float64
 5   distance_km         1188 non-null   float64
 6   declared_value      1212 non-null   object 
 7   n_stops             1224 non-null   int64  
 8   rain_flag           1224 non-null   int64  
 9   sale_event_flag     1224 non-null   int64  
 10  dispatch_time       1224 non-null   object 
 11  label_print_batch   1224 non-null   int64  
 12  scanner_build_no    1224 non-null   int64  
 13  tape_roll_no        1224 non-null   int64  
 14  sticker_serial_mod  1224 non-null   float64
 15  shelf_slot_no       1224 non-null   int64  
 16  delive

In [ ]:
missing_counts = raw_ship_df.isna().sum()

print(missing_counts[missing_counts > 0])

payment_mode        18
parcel_weight_kg    25
distance_km         36
declared_value      12
dtype: int64


In [ ]:
len(raw_ship_df.drop_duplicates())

1200

In [ ]:
len(raw_ship_df)

1224

In [ ]:
shipments = raw_ship_df.drop_duplicates()
shipments

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,"₹1,036",90,0,0,2026-09-12 10:48,7591,36,344,0.383,192,66.44
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,"₹1,122",51,0,0,2026-07-21 08:39,9701,21,413,0.595,102,64.30
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,₹288,76,0,0,2026-08-09 15:41,7098,31,448,0.854,97,72.10
1198,SHP11199,HUB11,Express,COD,1.71,284.6,"₹4,636",73,0,0,2026-07-25 11:37,9966,9,67,0.179,130,27.55


In [ ]:
raw_ship_df.shape

(1224, 17)

In [ ]:
shipments["service_type"].unique()

array(['express', 'Standard', 'Economy', 'Express', 'STANDARD',
       '  Express ', '  Standard ', '  Economy ', 'ECONOMY', 'EXPRESS',
       'standard', 'economy'], dtype=object)

In [ ]:
shipments["service_type"] = (shipments["service_type"].str.strip().str.title())

/tmp/ipykernel_10043/3611932710.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shipments["service_type"] = (shipments["service_type"].str.strip().str.title())


In [ ]:
shipments["service_type"].unique()

array(['Express', 'Standard', 'Economy'], dtype=object)

In [ ]:
shipments["service_type"].value_counts()

,count
service_type,
Standard,642
Economy,315
Express,243


In [ ]:
shipments["service_type"].value_counts()*100/len(shipments)

,count
service_type,
Standard,53.50
Economy,26.25
Express,20.25


In [ ]:
shipments["service_type"].value_counts(normalize = True)*100

,proportion
service_type,
Standard,53.50
Economy,26.25
Express,20.25


In [ ]:
shipments["payment_mode"].isna().sum()

np.int64(18)

In [ ]:
shipments["payment_mode"].value_counts()

,count
payment_mode,
Prepaid,729
COD,453


In [ ]:
len(shipments)

1200

In [ ]:
shipments["payment_mode"] = (
    shipments["payment_mode"]
    .fillna("Unknown")
)

/tmp/ipykernel_10043/3024754691.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shipments["payment_mode"] = (


In [ ]:
shipments["payment_mode"].value_counts()

,count
payment_mode,
Prepaid,729
COD,453
Unknown,18


In [ ]:
shipments["declared_value_inr"] = (
    shipments["declared_value"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

/tmp/ipykernel_10043/4189720518.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shipments["declared_value_inr"] = (


In [ ]:
shipments = shipments.drop(
    columns=["declared_value"]
)

In [ ]:
shipments

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79,1290.0
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56,1030.0
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56,441.0
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18,3122.0
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32,276.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,90,0,0,2026-09-12 10:48,7591,36,344,0.383,192,66.44,1036.0
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,51,0,0,2026-07-21 08:39,9701,21,413,0.595,102,64.30,1122.0
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,76,0,0,2026-08-09 15:41,7098,31,448,0.854,97,72.10,288.0
1198,SHP11199,HUB11,Express,COD,1.71,284.6,73,0,0,2026-07-25 11:37,9966,9,67,0.179,130,27.55,4636.0


In [ ]:
shipments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1200 entries, 0 to 1199
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   shipment_id         1200 non-null   object 
 1   hub_id              1200 non-null   object 
 2   service_type        1200 non-null   object 
 3   payment_mode        1200 non-null   object 
 4   parcel_weight_kg    1176 non-null   float64
 5   distance_km         1164 non-null   float64
 6   n_stops             1200 non-null   int64  
 7   rain_flag           1200 non-null   int64  
 8   sale_event_flag     1200 non-null   int64  
 9   dispatch_time       1200 non-null   object 
 10  label_print_batch   1200 non-null   int64  
 11  scanner_build_no    1200 non-null   int64  
 12  tape_roll_no        1200 non-null   int64  
 13  sticker_serial_mod  1200 non-null   float64
 14  shelf_slot_no       1200 non-null   int64  
 15  delivery_hours      1200 non-null   float64
 16  declared_va

In [ ]:
shipments["dispatch_time"] = pd.to_datetime(
    shipments["dispatch_time"]
)
shipments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1200 entries, 0 to 1199
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   shipment_id         1200 non-null   object        
 1   hub_id              1200 non-null   object        
 2   service_type        1200 non-null   object        
 3   payment_mode        1200 non-null   object        
 4   parcel_weight_kg    1176 non-null   float64       
 5   distance_km         1164 non-null   float64       
 6   n_stops             1200 non-null   int64         
 7   rain_flag           1200 non-null   int64         
 8   sale_event_flag     1200 non-null   int64         
 9   dispatch_time       1200 non-null   datetime64[ns]
 10  label_print_batch   1200 non-null   int64         
 11  scanner_build_no    1200 non-null   int64         
 12  tape_roll_no        1200 non-null   int64         
 13  sticker_serial_mod  1200 non-null   float64       
 1

In [ ]:
sentinel_count = (
    shipments["parcel_weight_kg"] > 100
).sum()

shipments.loc[
    shipments["parcel_weight_kg"] > 100,
    "parcel_weight_kg"
] = np.nan

In [ ]:
toy_weights = pd.Series([
    1.0, 2.0, np.nan, 3.0, 4.0
])

print("Mean:", toy_weights.mean())
print("Median:", toy_weights.median())

filled = toy_weights.fillna(
    toy_weights.median()
)

print(filled)

Mean: 2.5
Median: 2.5
0    1.0
1    2.0
2    2.5
3    3.0
4    4.0
dtype: float64


In [ ]:
toy_weights = pd.Series([
    1.0, 2.0, np.nan, 3.0, 4.0
])

print("Mean:", toy_weights.mean())
print("Median:", toy_weights.median())

Mean: 2.5
Median: 2.5


In [ ]:
toy_weights = pd.Series([
    1.0, 2.0, np.nan, 3.0, 4.0, 100.0
])

print("Mean:", toy_weights.mean())
print("Median:", toy_weights.median())

Mean: 22.0
Median: 3.0


In [ ]:
medians = {
    "distance_km":
        round(float(shipments["distance_km"].median()),2),

    "parcel_weight_kg":
        round(float(shipments["parcel_weight_kg"].median()),2),

    "declared_value_inr":
        round(float(shipments["declared_value_inr"].median()),2)
}

print(medians)

{'distance_km': 314.95, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}


In [ ]:
import numpy
import pandas
import sklearn
import joblib

In [ ]:
print("Pandas:", pd.__version__)

Pandas: 2.2.3


In [ ]:
print("Numpy:", np.__version__)

Numpy: 2.4.6


In [ ]:
print("Scikit-learn:", sklearn.__version__)

Scikit-learn: 1.6.1


In [ ]:
print("Joblib:", joblib.__version__)

Joblib: 1.6.0


In [ ]:
times = pd.Series([
    "2026-07-01 09:30",
    "2026-07-04 18:45"
])
times

,0
0,2026-07-01 09:30
1,2026-07-04 18:45


In [ ]:
times = pd.to_datetime(times)

print(times)
print(times.dt.hour)

0   2026-07-01 09:30:00
1   2026-07-04 18:45:00
dtype: datetime64[ns]
0     9
1    18
dtype: int32


In [ ]:
times.dt.dayofweek

,0
0,2
1,5


In [ ]:
weights = pd.Series([
    1.5, 2.0, 4.2, 999.0, 3.1
])

weights[weights > 100] = np.nan
weights

,0
0,1.5
1,2.0
2,4.2
3,NaN
4,3.1


In [ ]:
medians

{'distance_km': 314.95, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}

In [ ]:
for k in medians.keys():
  print(k)

distance_km
parcel_weight_kg
declared_value_inr


In [ ]:
for v in medians.values():
  print(v)

314.95
1.83
1121.5


In [ ]:
for i in medians.items():
  print(i)

('distance_km', 314.95)
('parcel_weight_kg', 1.83)
('declared_value_inr', 1121.5)


In [ ]:
for i in medians.items():
  print("Key", i[0], ", value:", i[1])

Key distance_km , value: 314.95
Key parcel_weight_kg , value: 1.83
Key declared_value_inr , value: 1121.5


In [ ]:
for i in medians.items():
  k,v = i
  print(f"key : {k}, value : {v}")

key : distance_km, value : 314.95
key : parcel_weight_kg, value : 1.83
key : declared_value_inr, value : 1121.5


In [ ]:
for k,v in medians.items():
  print(f"key : {k}, value : {v}")

key : distance_km, value : 314.95
key : parcel_weight_kg, value : 1.83
key : declared_value_inr, value : 1121.5


In [ ]:
for column, median_value in medians.items():
    shipments[column] = (
        shipments[column].fillna(median_value)
    )

In [ ]:
  shipments[
        [
            "distance_km",
            "parcel_weight_kg",
            "declared_value_inr"
        ]
    ].isna().sum()

,0
distance_km,0
parcel_weight_kg,0
declared_value_inr,0


In [ ]:
hubs_df

,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300
5,HUB06,Pune,West,1,1250
6,HUB07,Kolkata,East,1,1200
7,HUB08,Ahmedabad,West,1,1150
8,HUB09,Jaipur,North,2,950
9,HUB10,Lucknow,North,2,900


In [ ]:
hubs_df.head()

,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300


In [ ]:
toy_shipments = pd.DataFrame({
    "shipment": ["A", "B", "C"],
    "hub_id": ["HUB01", "HUB02", "HUB01"]
})

toy_hubs = pd.DataFrame({
    "hub_id": ["HUB01", "HUB02"],
    "hub_city": ["Mumbai", "Delhi"]
})

In [ ]:
toy_shipments

,shipment,hub_id
0,A,HUB01
1,B,HUB02
2,C,HUB01


In [ ]:
toy_hubs

,hub_id,hub_city
0,HUB01,Mumbai
1,HUB02,Delhi


In [ ]:
merged = toy_shipments.merge(
    toy_hubs,
    on="hub_id",
    how="left"
)

In [ ]:
merged

,shipment,hub_id,hub_city
0,A,HUB01,Mumbai
1,B,HUB02,Delhi
2,C,HUB01,Mumbai


In [ ]:
PROMISED_HOURS = {
    "Express": 36,
    "Standard": 60,
    "Economy": 96
}

FEATURES = [
    "promised_hours",
    "distance_km",
    "parcel_weight_kg",
    "log_declared_value",
    "n_stops",
    "rain_flag",
    "sale_event_flag",
    "is_cod",
    "is_weekend",
    "dispatch_hour",
    "hub_tier",
    "hub_daily_capacity",
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

NOISE_COLUMNS = [
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

In [ ]:
 result = shipments.merge(
        hubs[
            [
                "hub_id",
                "hub_city",
                "region",
                "hub_tier",
                "hub_daily_capacity"
            ]
        ],
        on="hub_id",
        how="left",
        validate="many_to_one"
    )

In [ ]:
result.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,...,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr,hub_city,region,hub_tier,hub_daily_capacity
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,...,25,167,0.650,13,34.79,1290.0,Lucknow,North,2,900
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,...,31,169,0.045,243,58.56,1030.0,Lucknow,North,2,900
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,...,25,170,0.915,80,76.56,441.0,Bhubaneswar,East,3,390
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,...,21,338,0.380,175,41.18,3122.0,Bengaluru,South,1,1450
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,...,34,215,0.824,162,47.32,276.0,Coimbatore,South,2,740


In [ ]:
  result["promised_hours"] = (
        result["service_type"]
        .map(PROMISED_HOURS)
        .astype(int)
    )
  result

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,...,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr,hub_city,region,hub_tier,hub_daily_capacity,promised_hours
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,...,167,0.650,13,34.79,1290.0,Lucknow,North,2,900,36
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,...,169,0.045,243,58.56,1030.0,Lucknow,North,2,900,60
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,...,170,0.915,80,76.56,441.0,Bhubaneswar,East,3,390,96
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,...,338,0.380,175,41.18,3122.0,Bengaluru,South,1,1450,60
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,...,215,0.824,162,47.32,276.0,Coimbatore,South,2,740,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,90,0,0,2026-09-12 10:48:00,...,344,0.383,192,66.44,1036.0,Guwahati,East,3,450,60
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,51,0,0,2026-07-21 08:39:00,...,413,0.595,102,64.30,1122.0,Chennai,South,1,1300,96
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,76,0,0,2026-08-09 15:41:00,...,448,0.854,97,72.10,288.0,Chandigarh,North,2,850,96
1198,SHP11199,HUB11,Express,COD,1.71,284.6,73,0,0,2026-07-25 11:37:00,...,67,0.179,130,27.55,4636.0,Chandigarh,North,2,850,36


In [ ]:
    result["is_cod"] = (
        result["payment_mode"] == "COD"
    ).astype(int)

In [ ]:
   result["is_weekend"] = (
        result["dispatch_time"].dt.dayofweek >= 5
    ).astype(int)

In [ ]:
   result["dispatch_hour"] = (
        result["dispatch_time"].dt.hour
    ).astype(int)

In [ ]:
result.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,...,delivery_hours,declared_value_inr,hub_city,region,hub_tier,hub_daily_capacity,promised_hours,is_cod,is_weekend,dispatch_hour
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,...,34.79,1290.0,Lucknow,North,2,900,36,1,1,19
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,...,58.56,1030.0,Lucknow,North,2,900,60,1,1,7
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,...,76.56,441.0,Bhubaneswar,East,3,390,96,0,1,12
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,...,41.18,3122.0,Bengaluru,South,1,1450,60,0,0,17
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,...,47.32,276.0,Coimbatore,South,2,740,60,0,1,17


In [ ]:
import os

for folder in ["data", "src", "models", "outputs"]:
    print(f"\n{folder}/")

    if os.path.exists(folder):
        print(os.listdir(folder))
    else:
        print("Folder does not exist")


data/
['hubs.csv', 'raw_shipments.csv', 'new_shipments.csv']

src/
Folder does not exist

models/
Folder does not exist

outputs/
[]


In [ ]:
import os

for folder in ["src", "models", "outputs"]:
    os.makedirs(folder, exist_ok=True)

print("Project structure:")
for folder in ["data", "src", "models", "outputs"]:
    print(folder, "->", os.listdir(folder))

Project structure:
data -> ['hubs.csv', 'raw_shipments.csv', 'new_shipments.csv']
src -> []
models -> []
outputs -> []


In [ ]:
import pandas as pd
import numpy as np
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")
df = raw.drop_duplicates().copy()
df["service_type"] = df["service_type"].str.strip().str.title()
df["payment_mode"] = df["payment_mode"].fillna("Unknown")
df["declared_value_inr"] = pd.to_numeric(
    df["declared_value"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)
df = df.drop(columns=["declared_value"])
df["dispatch_time"] = pd.to_datetime(df["dispatch_time"])
df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan
medians = {
    col: float(df[col].median())
    for col in ["distance_km", "parcel_weight_kg", "declared_value_inr"]
}

for col, value in medians.items():
    df[col] = df[col].fillna(value)

df = df.merge(
    hubs[["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"]],
    on="hub_id",
    how="left"
)

PROMISED_HOURS = {
    "Express": 36,
    "Standard": 60,
    "Economy": 96
}

df["promised_hours"] = df["service_type"].map(PROMISED_HOURS)
df["is_cod"] = (df["payment_mode"] == "COD").astype(int)
df["is_weekend"] = (df["dispatch_time"].dt.dayofweek >= 5).astype(int)
df["dispatch_hour"] = df["dispatch_time"].dt.hour
df["log_declared_value"] = np.log(df["declared_value_inr"])
df["is_delayed"] = (df["delivery_hours"] > df["promised_hours"]).astype(int)
df.to_csv("data/modelling_table.csv", index=False)

print("Modelling table created!")
print("Rows:", len(df))
print("Medians:", medians)
print("Delayed shipments:", df["is_delayed"].sum())

Modelling table created!
Rows: 1200
Medians: {'distance_km': 314.95000000000005, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}
Delayed shipments: 278


In [ ]:
print(pd.read_csv("data/modelling_table.csv").shape)

(1200, 27)


In [ ]:
import pandas as pd
import os

os.makedirs("outputs", exist_ok=True)

df = pd.read_csv("data/modelling_table.csv")
df["distance_band"] = pd.cut(
    df["distance_km"],
    bins=[0, 200, 400, 600, 2000],
    labels=["0-200", "201-400", "401-600", "601+"],
    right=True
)

def calculate_delay_evidence(data, dimension):

    result = (
        data.groupby(dimension, observed=False)
        .agg(
            shipments=("delivery_hours", "count"),
            delayed=("is_delayed", "sum"),
            delay_rate=("is_delayed", "mean"),
            mean_hours=("delivery_hours", "mean")
        )
        .reset_index()
    )

    result.insert(0, "dimension", dimension)
    result.rename(columns={dimension: "group"}, inplace=True)

    return result
service = calculate_delay_evidence(df, "service_type")
tier = calculate_delay_evidence(df, "hub_tier")
distance = calculate_delay_evidence(df, "distance_band")
result = pd.concat(
    [service, tier, distance],
    ignore_index=True
)

result["delay_rate"] = result["delay_rate"].round(4)
result["mean_hours"] = result["mean_hours"].round(4)
result = result[
    ["dimension", "group", "shipments", "delayed", "delay_rate", "mean_hours"]
]

result.to_csv("outputs/part_a_delay_by_group.csv", index=False)

print("Delay evidence CSV created!")
display(result)

Delay evidence CSV created!


,dimension,group,shipments,delayed,delay_rate,mean_hours
0,service_type,Economy,315,27,0.0857,82.0931
1,service_type,Express,243,119,0.4897,36.8056
2,service_type,Standard,642,132,0.2056,52.2569
3,hub_tier,1,400,16,0.0400,48.5664
4,hub_tier,2,400,86,0.2150,58.8194
5,hub_tier,3,400,176,0.4400,63.4943
6,distance_band,0-200,274,6,0.0219,46.7559
7,distance_band,201-400,563,77,0.1368,55.9660
8,distance_band,401-600,275,120,0.4364,64.2660
9,distance_band,601+,88,75,0.8523,72.2608


In [ ]:
check = pd.read_csv("outputs/part_a_delay_by_group.csv")

print("Total rows:", len(check))
print("Total shipments:", check["shipments"].sum())
print("Dimensions:", check["dimension"].unique())
print("Columns:", check.columns.tolist())

Total rows: 10
Total shipments: 3600
Dimensions: ['service_type' 'hub_tier' 'distance_band']
Columns: ['dimension', 'group', 'shipments', 'delayed', 'delay_rate', 'mean_hours']


In [ ]:
import pandas as pd
import os

df = pd.read_csv("data/modelling_table.csv")

numeric_df = df.select_dtypes(include="number")
correlations = numeric_df.corr()
target_corr = correlations["delivery_hours"].drop("delivery_hours")
target_corr = target_corr.reindex(
    target_corr.abs().sort_values(ascending=False).index
)
correlation_output = target_corr.reset_index()
correlation_output.columns = ["feature", "correlation"]

correlation_output["correlation"] = (
    correlation_output["correlation"].round(4)
)

os.makedirs("outputs", exist_ok=True)

correlation_output.to_csv(
    "outputs/part_a_correlations.csv",
    index=False
)

print("Correlation analysis completed!")
display(correlation_output)

Correlation analysis completed!


,feature,correlation
0,promised_hours,0.8372
1,distance_km,0.4013
2,n_stops,0.3247
3,hub_daily_capacity,-0.3237
4,hub_tier,0.3175
5,sale_event_flag,0.1709
6,rain_flag,0.1437
7,is_delayed,0.1391
8,is_weekend,0.1038
9,is_cod,0.0610


In [ ]:
import pandas as pd
import numpy as np
import os

raw = pd.read_csv("data/raw_shipments.csv")
df = pd.read_csv("data/modelling_table.csv")

checks = []

def add_check(name, actual, expected, status):
    checks.append({
        "check": name,
        "actual": actual,
        "expected": expected,
        "status": status
    })
duplicates = raw.duplicated().sum()
add_check("Exact duplicate rows", duplicates, 24, duplicates == 24)
add_check("Rows after cleaning", len(df), 1200, len(df) == 1200)
missing = int(df.isnull().sum().sum())
add_check("Remaining missing values", missing, 0, missing == 0)
delayed = int(df["is_delayed"].sum())
add_check("Delayed shipments", delayed, 278, delayed == 278)
delay_rate = df["is_delayed"].mean()
add_check("Delay rate", round(delay_rate, 4), 0.2317,
          round(delay_rate, 4) == 0.2317)
service_count = df["service_type"].nunique()
add_check("Unique service types", service_count, 3, service_count == 3)
hub_count = df["hub_id"].nunique()
add_check("Unique hubs", hub_count, 24, hub_count == 24)
checks_df = pd.DataFrame(checks)
os.makedirs("outputs", exist_ok=True)

checks_df.to_csv("outputs/part_a_checks.csv", index=False)

display(checks_df)
print("\nAll checks passed:", checks_df["status"].all())

,check,actual,expected,status
0,Exact duplicate rows,24.0000,24.0000,True
1,Rows after cleaning,1200.0000,1200.0000,True
2,Remaining missing values,0.0000,0.0000,True
3,Delayed shipments,278.0000,278.0000,True
4,Delay rate,0.2317,0.2317,True
5,Unique service types,3.0000,3.0000,True
6,Unique hubs,24.0000,24.0000,True



All checks passed: True


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/modelling_table.csv")
FEATURES = [
    "promised_hours",
    "distance_km",
    "parcel_weight_kg",
    "log_declared_value",
    "n_stops",
    "rain_flag",
    "sale_event_flag",
    "is_cod",
    "is_weekend",
    "dispatch_hour",
    "hub_tier",
    "hub_daily_capacity",
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]
X = df[FEATURES]
y = df["delivery_hours"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Testing rows:", len(X_test))
print("Number of features:", len(FEATURES))

Training rows: 840
Validation rows: 180
Testing rows: 180
Number of features: 17


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
train_means = X_train_scaled.mean(axis=0)
val_means = X_val_scaled.mean(axis=0)

print("Maximum absolute training mean:",
      np.max(np.abs(train_means)))

print("Maximum absolute validation mean:",
      np.max(np.abs(val_means)))

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Maximum absolute training mean: 4.039097099112474e-16
Maximum absolute validation mean: 0.13148633082383607
Training shape: (840, 17)
Validation shape: (180, 17)
Testing shape: (180, 17)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
baseline_prediction = np.full(
    len(y_val),
    y_train.mean()
)
mae = mean_absolute_error(y_val, baseline_prediction)
mse = mean_squared_error(y_val, baseline_prediction)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, baseline_prediction)

print("BASELINE REGRESSION RESULTS")
print("---------------------------")
print("Training mean:", round(y_train.mean(), 4))
print("MAE:", round(mae, 4))
print("MSE:", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("R2 Score:", round(r2, 4))

BASELINE REGRESSION RESULTS
---------------------------
Training mean: 57.7493
MAE: 15.7952
MSE: 355.5204
RMSE: 18.8552
R2 Score: -0.0212


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import os

depths = [1, 2, 3, 4, 6, 8, 12, None]

results = []

for depth in depths:

    model = DecisionTreeRegressor(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    rmse = np.sqrt(
        mean_squared_error(y_val, predictions)
    )

    results.append({
        "max_depth": str(depth),
        "validation_rmse": rmse
    })
depth_results = pd.DataFrame(results)
depth_results["_sort_depth"] = depth_results["max_depth"].replace(
    "None", "999"
).astype(int)

depth_results = depth_results.sort_values(
    ["validation_rmse", "_sort_depth"]
).drop(columns="_sort_depth")
best_depth = depth_results.iloc[0]["max_depth"]

if best_depth == "None":
    best_depth = None
else:
    best_depth = int(best_depth)
os.makedirs("outputs", exist_ok=True)

depth_results.to_csv(
    "outputs/part_b_tree_depth.csv",
    index=False
)

print("Decision Tree depth results:")
display(depth_results)

print("Best depth:", best_depth)

Decision Tree depth results:


,max_depth,validation_rmse
4,6,6.279637
5,8,6.790075
6,12,7.353997
3,4,7.389621
7,None,7.836781
2,3,8.703332
1,2,10.411443
0,1,12.132363


Best depth: 6


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=6,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)
    mse = mean_squared_error(y_val, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, predictions)

    results.append({
        "model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

metrics_df = pd.DataFrame(results)

metrics_df = metrics_df.sort_values(
    "RMSE"
).reset_index(drop=True)
metrics_df.to_csv(
    "outputs/part_b_metrics.csv",
    index=False
)

display(metrics_df)

print("\nBest model:", metrics_df.iloc[0]["model"])

,model,MAE,MSE,RMSE,R2
0,Linear Regression,3.922162,23.923532,4.891169,0.931280
1,Gradient Boosting,3.916577,24.172216,4.916525,0.930566
2,Random Forest,4.288197,30.355005,5.509538,0.912806
3,Decision Tree,5.097064,39.433846,6.279637,0.886728



Best model: Linear Regression


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

learning_rates = [0.01, 0.1, 1.0]

gb_results = []

for rate in learning_rates:
    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=rate,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    mse = mean_squared_error(y_val, predictions)

    gb_results.append({
        "learning_rate": rate,
        "MAE": mean_absolute_error(y_val, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_val, predictions)
    })

gb_df = pd.DataFrame(gb_results).sort_values("RMSE")
gb_df.to_csv(
    "outputs/part_b_gb_learning_rate.csv",
    index=False
)

display(gb_df)

,learning_rate,MAE,MSE,RMSE,R2
1,0.10,3.916577,24.172216,4.916525,0.930566
2,1.00,5.717849,54.317869,7.370066,0.843974
0,0.01,7.976331,100.035174,10.001759,0.712652


In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
import pandas as pd
import numpy as np

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

cv_models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in cv_models.items():
    scores = cross_val_score(
        model,
        X_trainval,
        y_trainval,
        cv=kf,
        scoring="r2"
    )

    cv_results.append({
        "model": name,
        "mean_cv_r2": scores.mean(),
        "std_cv_r2": scores.std()
    })

cv_df = pd.DataFrame(cv_results)

cv_df.to_csv("outputs/part_b_cv.csv", index=False)

display(cv_df)

,model,mean_cv_r2,std_cv_r2
0,Linear Regression,0.926133,0.004754
1,Gradient Boosting,0.935266,0.004477


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

final_regression_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
final_regression_model.fit(X_trainval, y_trainval)
test_predictions = final_regression_model.predict(X_test)

test_mse = mean_squared_error(y_test, test_predictions)

test_results = pd.DataFrame([{
    "model": "Gradient Boosting",
    "MAE": mean_absolute_error(y_test, test_predictions),
    "MSE": test_mse,
    "RMSE": np.sqrt(test_mse),
    "R2": r2_score(y_test, test_predictions)
}])
test_results.to_csv(
    "outputs/part_b_final_test.csv",
    index=False
)

display(test_results)

,model,MAE,MSE,RMSE,R2
0,Gradient Boosting,3.426107,19.954647,4.467062,0.95051


In [ ]:
import pandas as pd

metrics = pd.read_csv("outputs/part_b_metrics.csv")

metrics.insert(0, "evaluation_set", "validation")
test_row = pd.DataFrame([{
    "evaluation_set": "test",
    "model": "Gradient Boosting",
    "MAE": 3.426107,
    "MSE": 19.954647,
    "RMSE": 4.467062,
    "R2": 0.950510
}])

metrics = pd.concat([metrics, test_row], ignore_index=True)
metrics.to_csv("outputs/part_b_metrics.csv", index=False)

checks = pd.DataFrame([
    {"check": "Training rows", "result": len(X_train), "passed": len(X_train) == 840},
    {"check": "Validation rows", "result": len(X_val), "passed": len(X_val) == 180},
    {"check": "Test rows", "result": len(X_test), "passed": len(X_test) == 180},
    {"check": "Feature count", "result": X_train.shape[1], "passed": X_train.shape[1] == 17},
    {"check": "Best validation model", "result": "Linear Regression", "passed": True},
    {"check": "Best cross-validation model", "result": "Gradient Boosting", "passed": True},
    {"check": "Cross-validation mean R2", "result": 0.935266, "passed": 0.935266 >= 0.90},
    {"check": "Cross-validation std R2", "result": 0.004477, "passed": 0.004477 <= 0.03},
    {"check": "Final test R2", "result": 0.950510, "passed": 0.950510 >= 0.88}
])

checks.to_csv("outputs/part_b_checks.csv", index=False)

display(metrics)
display(checks)

print("All Part B checks passed:", checks["passed"].all())

,evaluation_set,model,MAE,MSE,RMSE,R2
0,validation,Linear Regression,3.922162,23.923532,4.891169,0.931280
1,validation,Gradient Boosting,3.916577,24.172216,4.916525,0.930566
2,validation,Random Forest,4.288197,30.355005,5.509538,0.912806
3,validation,Decision Tree,5.097064,39.433846,6.279637,0.886728
4,test,Gradient Boosting,3.426107,19.954647,4.467062,0.950510


,check,result,passed
0,Training rows,840,True
1,Validation rows,180,True
2,Test rows,180,True
3,Feature count,17,True
4,Best validation model,Linear Regression,True
5,Best cross-validation model,Gradient Boosting,True
6,Cross-validation mean R2,0.935266,True
7,Cross-validation std R2,0.004477,True
8,Final test R2,0.95051,True


All Part B checks passed: True


In [ ]:
from sklearn.model_selection import train_test_split

X_cls = df[FEATURES]
y_cls = df["is_delayed"].astype(int)

Xc_train, Xc_temp, yc_train, yc_temp = train_test_split(
    X_cls,
    y_cls,
    test_size=0.30,
    random_state=42,
    stratify=y_cls
)

Xc_val, Xc_test, yc_val, yc_test = train_test_split(
    Xc_temp,
    yc_temp,
    test_size=0.50,
    random_state=42,
    stratify=yc_temp
)
print("Training rows:", len(Xc_train))
print("Validation rows:", len(Xc_val))
print("Testing rows:", len(Xc_test))
print("\nDelayed shipments:")
print("Training:", yc_train.sum())
print("Validation:", yc_val.sum())
print("Testing:", yc_test.sum())

Training rows: 840
Validation rows: 180
Testing rows: 180

Delayed shipments:
Training: 195
Validation: 42
Testing: 41


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_model(name, y_true, predictions):
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        "model": name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        )
    }

results = []

blind_predictions = np.zeros(len(yc_val), dtype=int)

results.append(
    evaluate_model("Blind Baseline", yc_val, blind_predictions)
)

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=1000))
])

logistic_model.fit(Xc_train, yc_train)
logistic_predictions = logistic_model.predict(Xc_val)

results.append(
    evaluate_model(
        "Logistic Regression",
        yc_val,
        logistic_predictions
    )
)

best_tree = None
best_depth = None
best_f1 = -1

for depth in [2, 3, 4, 5, 6, 8, 10]:

    tree = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    tree.fit(Xc_train, yc_train)

    predictions = tree.predict(Xc_val)

    score = f1_score(
        yc_val, predictions, zero_division=0
    )

    if score > best_f1:
        best_f1 = score
        best_tree = tree
        best_depth = depth

tree_predictions = best_tree.predict(Xc_val)

results.append(
    evaluate_model(
        f"Decision Tree (depth={best_depth})",
        yc_val,
        tree_predictions
    )
)

random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest.fit(Xc_train, yc_train)

rf_predictions = random_forest.predict(Xc_val)

results.append(
    evaluate_model(
        "Random Forest",
        yc_val,
        rf_predictions
    )
)

part_c_models = pd.DataFrame(results)

part_c_models.to_csv(
    "outputs/part_c_models.csv",
    index=False
)

display(part_c_models)

print("Best Decision Tree depth:", best_depth)

,model,TN,FP,FN,TP,accuracy,precision,recall,f1
0,Blind Baseline,138,0,42,0,0.766667,0.000000,0.000000,0.000000
1,Logistic Regression,128,10,7,35,0.905556,0.777778,0.833333,0.804598
2,Decision Tree (depth=6),128,10,11,31,0.883333,0.756098,0.738095,0.746988
3,Random Forest,132,6,11,31,0.905556,0.837838,0.738095,0.784810


Best Decision Tree depth: 6


In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

NOISE_COLUMNS = [
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

selector_all = SelectKBest(score_func=f_classif, k="all")

selector_all.fit(Xc_train, yc_train)

rf_all_features = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_all_features.fit(Xc_train, yc_train)

feature_scores = pd.DataFrame({
    "feature": FEATURES,
    "f_score": selector_all.scores_,
    "rf_importance": rf_all_features.feature_importances_
})

feature_scores["rank"] = (
    feature_scores["f_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

feature_scores["is_noise"] = (
    feature_scores["feature"].isin(NOISE_COLUMNS)
)

feature_scores = feature_scores.sort_values(
    "rank"
).reset_index(drop=True)

feature_scores.to_csv(
    "outputs/part_c_feature_scores.csv",
    index=False
)

display(feature_scores)

print("\nTop 7 features:")
print(feature_scores.head(7)["feature"].tolist())

,feature,f_score,rf_importance,rank,is_noise
0,distance_km,389.213126,0.270831,1,False
1,hub_tier,155.348313,0.039929,2,False
2,hub_daily_capacity,144.134684,0.081754,3,False
3,n_stops,125.479792,0.069879,4,False
4,promised_hours,84.417725,0.087394,5,False
5,rain_flag,53.032064,0.045019,6,False
6,sale_event_flag,36.728141,0.033463,7,False
7,label_print_batch,4.584717,0.045985,8,True
8,is_cod,3.894937,0.010593,9,False
9,is_weekend,2.693921,0.008696,10,False



Top 7 features:
['distance_km', 'hub_tier', 'hub_daily_capacity', 'n_stops', 'promised_hours', 'rain_flag', 'sale_event_flag']


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

k_values = [5, 7, 10, 12, 17]

selection_results = []

for k in k_values:

    selector = SelectKBest(score_func=f_classif, k=k)

    X_train_selected = selector.fit_transform(
        Xc_train, yc_train
    )

    X_val_selected = selector.transform(Xc_val)

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train_selected, yc_train)

    predictions = model.predict(X_val_selected)

    selected_features = [
        FEATURES[i]
        for i in selector.get_support(indices=True)
    ]

    selection_results.append({
        "k": k,
        "accuracy": accuracy_score(yc_val, predictions),
        "f1": f1_score(yc_val, predictions),
        "selected_features": ", ".join(selected_features)
    })

selectk_df = pd.DataFrame(selection_results)

selectk_df.to_csv(
    "outputs/part_c_selectk.csv",
    index=False
)

display(selectk_df)

,k,accuracy,f1,selected_features
0,5,0.861111,0.698795,"promised_hours, distance_km, n_stops, hub_tier..."
1,7,0.922222,0.833333,"promised_hours, distance_km, n_stops, rain_fla..."
2,10,0.900000,0.785714,"promised_hours, distance_km, n_stops, rain_fla..."
3,12,0.911111,0.804878,"promised_hours, distance_km, n_stops, rain_fla..."
4,17,0.905556,0.784810,"promised_hours, distance_km, parcel_weight_kg,..."


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import f1_score
import pandas as pd

selector_7 = SelectKBest(score_func=f_classif, k=7)
X_train_7 = selector_7.fit_transform(Xc_train, yc_train)
X_val_7 = selector_7.transform(Xc_val)

selected_features_7 = [
    FEATURES[i]
    for i in selector_7.get_support(indices=True)
]

print("Selected features:", selected_features_7)
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [4, 6, 8],
    "min_samples_leaf": [1, 5, 10]
}

default_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

default_rf.fit(X_train_7, yc_train)
default_predictions = default_rf.predict(X_val_7)
default_f1 = f1_score(yc_val, default_predictions)
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_7, yc_train)

grid_predictions = grid_search.best_estimator_.predict(X_val_7)

grid_f1 = f1_score(yc_val, grid_predictions)
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=9,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_7, yc_train)

random_predictions = random_search.best_estimator_.predict(X_val_7)

random_f1 = f1_score(yc_val, random_predictions)

tuning_results = pd.DataFrame([
    {
        "method": "default_forest_same_7_features",
        "candidate_count": 1,
        "fits": 1,
        "cv_accuracy": None,
        "validation_f1": default_f1
    },
    {
        "method": "grid_search",
        "candidate_count": 27,
        "fits": 135,
        "cv_accuracy": grid_search.best_score_,
        "validation_f1": grid_f1
    },
    {
        "method": "randomized_search",
        "candidate_count": 9,
        "fits": 45,
        "cv_accuracy": random_search.best_score_,
        "validation_f1": random_f1
    }
])

tuning_results.to_csv(
    "outputs/part_c_tuning.csv",
    index=False
)

display(tuning_results)

print("\nGrid Search best parameters:")
print(grid_search.best_params_)

print("\nRandomized Search best parameters:")
print(random_search.best_params_)

Selected features: ['promised_hours', 'distance_km', 'n_stops', 'rain_flag', 'sale_event_flag', 'hub_tier', 'hub_daily_capacity']


,method,candidate_count,fits,cv_accuracy,validation_f1
0,default_forest_same_7_features,1,1,NaN,0.833333
1,grid_search,27,135,0.895238,0.775000
2,randomized_search,9,45,0.890476,0.784810



Grid Search best parameters:
{'max_depth': 6, 'min_samples_leaf': 1, 'n_estimators': 100}

Randomized Search best parameters:
{'n_estimators': 50, 'min_samples_leaf': 1, 'max_depth': 6}


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

val_probabilities = default_rf.predict_proba(X_val_7)[:, 1]

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

threshold_results = []

for threshold in thresholds:

    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            yc_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            yc_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            yc_val, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)
eligible = threshold_df[
    threshold_df["recall"] >= 0.80
]

if len(eligible) > 0:
    best_threshold = eligible["threshold"].max()
else:
    best_threshold = 0.1

threshold_df.to_csv(
    "outputs/part_c_thresholds.csv",
    index=False
)

display(threshold_df)

print("\nSelected threshold:", best_threshold)

,threshold,precision,recall,f1
0,0.1,0.482353,0.976190,0.645669
1,0.2,0.573529,0.928571,0.709091
2,0.3,0.696429,0.928571,0.795918
3,0.4,0.740000,0.880952,0.804348
4,0.5,0.795455,0.833333,0.813953
5,0.6,0.888889,0.761905,0.820513
6,0.7,0.870968,0.642857,0.739726
7,0.8,0.826087,0.452381,0.584615
8,0.9,0.857143,0.285714,0.428571



Selected threshold: 0.5


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import joblib

FINAL_THRESHOLD = 0.5

final_pipeline = Pipeline([
    ("selector", SelectKBest(score_func=f_classif, k=7)),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

final_pipeline.fit(Xc_train, yc_train)
joblib.dump(final_pipeline, "models/delay_classifier.pkl")

print("Final classification pipeline trained and saved.")
print("Selected threshold:", FINAL_THRESHOLD)
print("Training rows:", len(Xc_train))

Final classification pipeline trained and saved.
Selected threshold: 0.5
Training rows: 840


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd
test_probabilities = final_pipeline.predict_proba(Xc_test)[:, 1]
test_predictions = (
    test_probabilities >= FINAL_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    yc_test, test_predictions, labels=[0, 1]
).ravel()
final_results = pd.DataFrame([{
    "threshold": FINAL_THRESHOLD,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
    "accuracy": accuracy_score(yc_test, test_predictions),
    "precision": precision_score(
        yc_test, test_predictions, zero_division=0
    ),
    "recall": recall_score(
        yc_test, test_predictions, zero_division=0
    ),
    "f1": f1_score(
        yc_test, test_predictions, zero_division=0
    )
}])
final_results.to_csv(
    "outputs/part_c_final_test.csv", index=False
)

display(final_results)

,threshold,TN,FP,FN,TP,accuracy,precision,recall,f1
0,0.5,125,14,10,31,0.866667,0.688889,0.756098,0.72093


In [ ]:
import pandas as pd

selected_features = [
    "promised_hours",
    "distance_km",
    "n_stops",
    "rain_flag",
    "sale_event_flag",
    "hub_tier",
    "hub_daily_capacity"
]

final_config = pd.DataFrame([{
    "model": "RandomForestClassifier",
    "n_estimators": 100,
    "random_state": 42,
    "selector": "SelectKBest",
    "k": 7,
    "threshold": FINAL_THRESHOLD,
    "selected_features": ", ".join(selected_features),
    "test_accuracy": final_results.loc[0, "accuracy"],
    "test_precision": final_results.loc[0, "precision"],
    "test_recall": final_results.loc[0, "recall"],
    "test_f1": final_results.loc[0, "f1"]
}])

final_config.to_csv(
    "outputs/part_c_final_config.csv",
    index=False
)

print("Final configuration saved successfully.")
display(final_config)

Final configuration saved successfully.


,model,n_estimators,random_state,selector,k,threshold,selected_features,test_accuracy,test_precision,test_recall,test_f1
0,RandomForestClassifier,100,42,SelectKBest,7,0.5,"promised_hours, distance_km, n_stops, rain_fla...",0.866667,0.688889,0.756098,0.72093


In [ ]:
hub_profile = (
    df.groupby("hub_id")
    .agg(
        shipments=("hub_id", "size"),
        mean_delivery_hours=("delivery_hours", "mean"),
        delay_rate=("is_delayed", "mean"),
        mean_distance_km=("distance_km", "mean"),
        mean_n_stops=("n_stops", "mean")
    )
    .reset_index()
    .sort_values("hub_id")
)
hub_profile.to_csv(
    "outputs/part_d_hub_profile.csv",
    index=False
)

print("Number of hubs:", len(hub_profile))
display(hub_profile.round(4))

Number of hubs: 24


,hub_id,shipments,mean_delivery_hours,delay_rate,mean_distance_km,mean_n_stops
0,HUB01,50,45.6850,0.02,185.061,54.70
1,HUB02,50,47.7904,0.04,195.130,54.50
2,HUB03,50,49.9894,0.04,200.033,55.68
3,HUB04,50,44.9116,0.04,191.363,55.72
4,HUB05,50,45.6334,0.00,184.236,53.90
5,HUB06,50,49.6766,0.02,182.640,56.14
6,HUB07,50,57.8536,0.14,304.736,70.76
7,HUB08,50,46.9910,0.02,184.044,56.94
8,HUB09,50,58.8520,0.14,304.627,70.10
9,HUB10,50,55.7860,0.14,306.130,70.48


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

CLUSTER_FEATURES = [
    "mean_delivery_hours",
    "delay_rate",
    "mean_distance_km",
    "mean_n_stops"
]
hub_scaler = StandardScaler()

hub_scaled = hub_scaler.fit_transform(
    hub_profile[CLUSTER_FEATURES]
)
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

hub_profile["cluster"] = kmeans.fit_predict(hub_scaled)
hub_profile.to_csv(
    "outputs/part_d_hub_clusters.csv",
    index=False
)

print("Clustering completed.")
print("Number of hubs:", len(hub_profile))
print("Number of clusters:", hub_profile["cluster"].nunique())

display(
    hub_profile[
        ["hub_id", "delay_rate", "mean_delivery_hours", "cluster"]
    ].round(4)
)

Clustering completed.
Number of hubs: 24
Number of clusters: 3


,hub_id,delay_rate,mean_delivery_hours,cluster
0,HUB01,0.02,45.6850,2
1,HUB02,0.04,47.7904,2
2,HUB03,0.04,49.9894,2
3,HUB04,0.04,44.9116,2
4,HUB05,0.00,45.6334,2
5,HUB06,0.02,49.6766,2
6,HUB07,0.14,57.8536,0
7,HUB08,0.02,46.9910,2
8,HUB09,0.14,58.8520,0
9,HUB10,0.14,55.7860,0


In [ ]:
cluster_summary = (
    hub_profile.groupby("cluster")
    .agg(
        hubs=("hub_id", "count"),
        mean_delay_rate=("delay_rate", "mean"),
        mean_delivery_hours=("mean_delivery_hours", "mean"),
        mean_distance_km=("mean_distance_km", "mean"),
        mean_n_stops=("mean_n_stops", "mean")
    )
    .reset_index()
    .sort_values("mean_delay_rate")
)

display(cluster_summary.round(4))

,cluster,hubs,mean_delay_rate,mean_delivery_hours,mean_distance_km,mean_n_stops
2,2,7,0.0257,47.2396,188.9296,55.3686
0,0,8,0.1550,56.8137,314.1762,70.7175
1,1,9,0.4600,64.6504,470.4757,84.3444


In [ ]:
tier_to_segment = {
    1: "Standard",
    2: "Watch",
    3: "Escalate"
}
hub_profile["expected_segment"] = (
    hub_profile["hub_tier"].map(tier_to_segment)
)

hub_profile["is_mismatch"] = (
    hub_profile["segment"] != hub_profile["expected_segment"]
)
tier_mismatch = hub_profile.loc[
    hub_profile["is_mismatch"],
    [
        "hub_id",
        "hub_tier",
        "segment",
        "expected_segment",
        "delay_rate",
        "mean_delivery_hours"
    ]
].copy()

tier_mismatch.to_csv(
    "outputs/tier_vs_segment_mismatch.csv",
    index=False
)

print("Number of mismatched hubs:", len(tier_mismatch))
display(tier_mismatch.round(4))

Number of mismatched hubs: 4


,hub_id,hub_tier,segment,expected_segment,delay_rate,mean_delivery_hours
6,HUB07,1,Watch,Standard,0.14,57.8536
12,HUB13,2,Escalate,Watch,0.46,66.0742
14,HUB15,2,Escalate,Watch,0.36,62.9694
21,HUB22,3,Watch,Escalate,0.20,55.1442


In [ ]:
hub_tiers = (
    df.groupby("hub_id")["hub_tier"]
    .first()
    .reset_index()
)
hub_profile = hub_profile.drop(
    columns=["hub_tier"],
    errors="ignore"
)

hub_profile = hub_profile.merge(
    hub_tiers,
    on="hub_id",
    how="left"
)

print("Hub tier added successfully.")
print("Missing hub tiers:", hub_profile["hub_tier"].isna().sum())

display(
    hub_profile[["hub_id", "hub_tier", "segment"]].head()
)

Hub tier added successfully.
Missing hub tiers: 0


,hub_id,hub_tier,segment
0,HUB01,1,Standard
1,HUB02,1,Standard
2,HUB03,1,Standard
3,HUB04,1,Standard
4,HUB05,1,Standard


In [ ]:
from sklearn.metrics import pairwise_distances
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

separation_results = []

for k in [2, 3, 4, 5]:

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = km.fit_predict(hub_scaled)
    within_distances = []

    for cluster_id in range(k):
        members = hub_scaled[labels == cluster_id]

        if len(members) > 1:
            distances = pairwise_distances(members)
            within_distances.extend(
                distances[np.triu_indices(len(members), k=1)]
            )

    within_mean = np.mean(within_distances)
    cluster_centers = km.cluster_centers_

    between_distances = [
        np.linalg.norm(cluster_centers[i] - cluster_centers[j])
        for i, j in combinations(range(k), 2)
    ]

    between_mean = np.mean(between_distances)

    sizes = pd.Series(labels).value_counts().sort_values(
        ascending=False
    )

    separation_results.append({
        "k": k,
        "cluster_sizes": "-".join(map(str, sizes.tolist())),
        "between_mean": between_mean,
        "within_mean": within_mean,
        "separation_ratio": between_mean / within_mean
    })

separation_df = pd.DataFrame(separation_results)

separation_df.to_csv(
    "outputs/part_d_separation.csv",
    index=False
)

display(separation_df.round(4))

,k,cluster_sizes,between_mean,within_mean,separation_ratio
0,2,15-9,3.5956,1.2620,2.8490
1,3,9-8-7,3.1976,0.6018,5.3136
2,4,8-7-5-4,2.9438,0.4809,6.1214
3,5,8-7-4-3-2,2.7804,0.4320,6.4368


In [ ]:
import os

for folder in ["models", "outputs", "src", "data"]:
    os.makedirs(folder, exist_ok=True)

print("MODEL FILES:")
print(os.listdir("models"))

print("\nOUTPUT FILES:")
print(os.listdir("outputs"))

MODEL FILES:
['delay_classifier.pkl']

OUTPUT FILES:
['part_c_final_config.csv', 'part_a_checks.csv', 'part_c_thresholds.csv', 'part_b_tree_depth.csv', 'part_c_models.csv', 'part_c_tuning.csv', 'part_b_cv.csv', 'part_b_final_test.csv', 'part_a_delay_by_group.csv', 'tier_vs_segment_mismatch.csv', 'part_d_hub_clusters.csv', 'hub_segments.csv', 'part_a_correlations.csv', 'part_c_feature_scores.csv', 'part_b_checks.csv', 'part_b_metrics.csv', 'segment_profile.csv', 'part_c_final_test.csv', 'part_d_separation.csv', 'part_b_gb_learning_rate.csv', 'part_d_hub_profile.csv', 'part_c_selectk.csv']


In [ ]:
print("Regression model variables:")

for name in [
    "final_model",
    "best_model",
    "final_regressor",
    "best_gb",
    "gb_model"
]:
    if name in globals():
        print(name, "is available:", type(globals()[name]))

Regression model variables:


In [ ]:
print("Available regression-related variables:")

for name in sorted(globals()):
    if any(word in name.lower() for word in [
        "regress", "gradient", "boost", "model", "trainval"
    ]):
        value = globals()[name]
        print(name, ":", type(value).__name__)

Available regression-related variables:
DecisionTreeRegressor : ABCMeta
GradientBoostingRegressor : ABCMeta
LinearRegression : ABCMeta
LogisticRegression : type
RandomForestRegressor : ABCMeta
X_trainval : DataFrame
cv_models : dict
evaluate_model : function
final_regression_model : GradientBoostingRegressor
logistic_model : Pipeline
model : RandomForestClassifier
models : dict
part_c_models : DataFrame
y_trainval : Series


In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)
joblib.dump(
    final_regression_model,
    "models/delay_model.pkl"
)

print("Regression model saved successfully.")

print("Saved model files:")
print(os.listdir("models"))

Regression model saved successfully.
Saved model files:
['delay_model.pkl', 'delay_classifier.pkl']


In [ ]:
print("Variables containing median information:")

for name in sorted(globals()):
    if "median" in name.lower():
        print(name, ":", type(globals()[name]).__name__)


Variables containing median information:
median_value : float
medians : dict


In [ ]:
print("Original preprocessing medians:")

print("Type:", type(medians).__name__)
print("Number of entries:", len(medians))

for column, value in medians.items():
    print(f"{column}: {value}")

Original preprocessing medians:
Type: dict
Number of entries: 3
distance_km: 314.95000000000005
parcel_weight_kg: 1.83
declared_value_inr: 1121.5


In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)
FEATURES = [
    "promised_hours",
    "distance_km",
    "parcel_weight_kg",
    "log_declared_value",
    "n_stops",
    "rain_flag",
    "sale_event_flag",
    "is_cod",
    "is_weekend",
    "dispatch_hour",
    "hub_tier",
    "hub_daily_capacity",
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

model_config = {
    "regression_model": "models/delay_model.pkl",
    "classification_model": "models/delay_classifier.pkl",
    "classification_threshold": 0.5,
    "classification_features": [
        "promised_hours",
        "distance_km",
        "n_stops",
        "rain_flag",
        "sale_event_flag",
        "hub_tier",
        "hub_daily_capacity"
    ],
    "feature_order": FEATURES,
    "medians": medians,
    "selector_k": 7,
    "classification_n_estimators": 100,
    "random_state": 42
}

joblib.dump(model_config, "models/model_config.pkl")

print("Model configuration saved successfully.")

print("\nSaved model files:")
print(os.listdir("models"))

Model configuration saved successfully.

Saved model files:
['delay_model.pkl', 'delay_classifier.pkl', 'model_config.pkl']


In [ ]:
import joblib
import numpy as np
import pandas as pd
loaded_classifier = joblib.load(
    "models/delay_classifier.pkl"
)
original_probabilities = final_pipeline.predict_proba(
    Xc_test
)[:, 1]
reloaded_probabilities = loaded_classifier.predict_proba(
    Xc_test
)[:, 1]
max_difference = np.max(
    np.abs(
        original_probabilities - reloaded_probabilities
    )
)

reload_results = pd.DataFrame([{
    "test_rows": len(Xc_test),
    "max_probability_difference": max_difference,
    "reload_check_passed": max_difference <= 1e-9
}])

reload_results.to_csv(
    "outputs/part_d_reload_check.csv",
    index=False
)

display(reload_results)

print("Reload verification completed.")

,test_rows,max_probability_difference,reload_check_passed
0,180,0.0,True


Reload verification completed.


In [ ]:
print("Raw shipment columns:")
print(raw.columns.tolist())

print("\nHub columns:")
print(hubs.columns.tolist())

print("\nModelling table columns:")
print(df.columns.tolist())

Raw shipment columns:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours']

Hub columns:
['hub_id', 'hub_city', 'region', 'hub_tier', 'hub_daily_capacity']

Modelling table columns:
['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'parcel_weight_kg', 'distance_km', 'n_stops', 'rain_flag', 'sale_event_flag', 'dispatch_time', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no', 'delivery_hours', 'declared_value_inr', 'hub_city', 'region', 'hub_tier', 'hub_daily_capacity', 'promised_hours', 'is_cod', 'is_weekend', 'dispatch_hour', 'log_declared_value', 'is_delayed']


In [ ]:
print("Service type and promised hours mapping:")

print(
    df[["service_type", "promised_hours"]]
    .drop_duplicates()
    .sort_values("service_type")
    .to_string(index=False)
)

Service type and promised hours mapping:
service_type  promised_hours
     Economy              96
     Express              36
    Standard              60


In [ ]:
import os
os.makedirs("src", exist_ok=True)

print("src folder is ready.")

src folder is ready.


In [ ]:


import pandas as pd
import numpy as np

PROMISED_HOURS = {
    "Economy": 96,
    "Express": 36,
    "Standard": 60
}

FEATURES = [
    "promised_hours",
    "distance_km",
    "parcel_weight_kg",
    "log_declared_value",
    "n_stops",
    "rain_flag",
    "sale_event_flag",
    "is_cod",
    "is_weekend",
    "dispatch_hour",
    "hub_tier",
    "hub_daily_capacity",
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

NOISE_COLUMNS = [
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]


def clean_shipments(raw, medians=None):

    data = raw.copy()
    data = data.drop_duplicates().copy()
    data["service_type"] = (
        data["service_type"]
        .astype("string")
        .str.strip()
        .str.title()
    )
    data["payment_mode"] = (
        data["payment_mode"].fillna("Unknown")
    )
    data["declared_value_inr"] = pd.to_numeric(
        data["declared_value"]
        .astype("string")
        .str.replace(r"[₹,]", "", regex=True),
        errors="coerce"
    )

    data = data.drop(columns=["declared_value"])
    data["dispatch_time"] = pd.to_datetime(
        data["dispatch_time"],
        errors="coerce"
    )
    data.loc[
        data["parcel_weight_kg"] > 100,
        "parcel_weight_kg"
    ] = np.nan
    numeric_columns = [
        "distance_km",
        "parcel_weight_kg",
        "declared_value_inr"
    ]

    if medians is None:
        medians_used = {
            column: data[column].median()
            for column in numeric_columns
        }
    else:
        medians_used = medians.copy()
    for column in numeric_columns:
        data[column] = data[column].fillna(
            medians_used[column]
        )

    return data, medians_used


def build_features(clean, hubs):

    data = clean.copy()
    data = data.merge(
        hubs[
            [
                "hub_id",
                "hub_city",
                "region",
                "hub_tier",
                "hub_daily_capacity"
            ]
        ],
        on="hub_id",
        how="left",
        validate="many_to_one"
    )
    data["promised_hours"] = (
        data["service_type"].map(PROMISED_HOURS)
    )
    data["is_cod"] = (
        data["payment_mode"]
        .astype(str)
        .str.upper()
        .eq("COD")
        .astype(int)
    )

    data["is_weekend"] = (
        data["dispatch_time"].dt.dayofweek >= 5
    ).astype(int)

    data["dispatch_hour"] = (
        data["dispatch_time"].dt.hour
    )
    data["log_declared_value"] = np.log1p(
        data["declared_value_inr"]
    )
    if "delivery_hours" in data.columns:
        data["is_delayed"] = (
            data["delivery_hours"] >
            data["promised_hours"]
        ).astype(int)

    return data

In [ ]:
import os
os.makedirs("src", exist_ok=True)

print("src folder is ready.")

src folder is ready.


In [ ]:
%%writefile src/shipment_prep.py

import pandas as pd
import numpy as np

PROMISED_HOURS = {
    "Economy": 96,
    "Express": 36,
    "Standard": 60
}

FEATURES = [
    "promised_hours",
    "distance_km",
    "parcel_weight_kg",
    "log_declared_value",
    "n_stops",
    "rain_flag",
    "sale_event_flag",
    "is_cod",
    "is_weekend",
    "dispatch_hour",
    "hub_tier",
    "hub_daily_capacity",
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

NOISE_COLUMNS = [
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]


def clean_shipments(raw, medians=None):
    data = raw.copy()

    data = data.drop_duplicates().copy()

    data["service_type"] = (
        data["service_type"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    data["payment_mode"] = data["payment_mode"].fillna("Unknown")

    data["declared_value_inr"] = pd.to_numeric(
        data["declared_value"].astype("string")
        .str.replace(r"[₹,]", "", regex=True),
        errors="coerce"
    )

    data = data.drop(columns=["declared_value"])

    data["dispatch_time"] = pd.to_datetime(
        data["dispatch_time"],
        errors="coerce"
    )

    data.loc[
        data["parcel_weight_kg"] > 100,
        "parcel_weight_kg"
    ] = np.nan

    numeric_columns = [
        "distance_km",
        "parcel_weight_kg",
        "declared_value_inr"
    ]

    if medians is None:
        medians_used = {
            col: data[col].median()
            for col in numeric_columns
        }
    else:
        medians_used = medians.copy()

    for col in numeric_columns:
        data[col] = data[col].fillna(medians_used[col])

    return data, medians_used


def build_features(clean, hubs):
    data = clean.copy()

    data = data.merge(
        hubs[
            [
                "hub_id",
                "hub_city",
                "region",
                "hub_tier",
                "hub_daily_capacity"
            ]
        ],
        on="hub_id",
        how="left",
        validate="many_to_one"
    )

    data["promised_hours"] = data["service_type"].map(
        PROMISED_HOURS
    )

    data["is_cod"] = (
        data["payment_mode"]
        .astype(str)
        .str.upper()
        .eq("COD")
        .astype(int)
    )

    data["is_weekend"] = (
        data["dispatch_time"].dt.dayofweek >= 5
    ).astype(int)

    data["dispatch_hour"] = data["dispatch_time"].dt.hour

    data["log_declared_value"] = np.log1p(
        data["declared_value_inr"]
    )

    if "delivery_hours" in data.columns:
        data["is_delayed"] = (
            data["delivery_hours"] > data["promised_hours"]
        ).astype(int)


    return data

Writing src/shipment_prep.py


In [ ]:
import os

print("Current working directory:", os.getcwd())
print("\nFiles and folders:")
print(os.listdir())

Current working directory: /content

Files and folders:
['.config', 'src', 'sample_data']


In [ ]:
import os

for folder in ["src", "data", "models", "outputs"]:
    print(f"\n{folder}:")
    if os.path.exists(folder):
        print(os.listdir(folder))
    else:

        print("Folder not found")


src:
['__pycache__', 'shipment_prep.py']

data:
Folder not found

models:
Folder not found

outputs:
Folder not found


In [ ]:
import os

for folder in ["data", "models", "outputs", "src"]:
    os.makedirs(folder, exist_ok=True)
print("All project folders are ready.")

All project folders are ready.


In [ ]:
import os

for file in [
    "data/raw_shipments.csv",
    "data/hubs.csv",
    "data/new_shipments.csv"
]:
    print(file, "->", "FOUND" if os.path.exists(file) else "MISSING")

data/raw_shipments.csv -> MISSING
data/hubs.csv -> MISSING
data/new_shipments.csv -> MISSING
